# LSTM baseline on the NTM copy task

First: **Runtime → Change runtime type → T4 GPU**, then run the cells in order.

In [ ]:
# Safe to re-run: clones the repo once, then just pulls updates
%cd /content
!nvidia-smi --query-gpu=name --format=csv
!git clone https://github.com/mgupta8143/neural-turing-machines.git 2>/dev/null || git -C neural-turing-machines pull
%cd /content/neural-turing-machines

Stop any training run that's already going, so two runs don't write to the same files.

In [ ]:
!pkill -f "main.py train" || true

Pick the model to train. `lstm` is the baseline; `ntm-ff` and `ntm-lstm` are the NTM with a feed-forward or LSTM controller. Each model keeps its own log, checkpoint and figures, so you can train all three and compare.

In [ ]:
MODEL = "ntm-ff"  # "lstm" | "ntm-ff" | "ntm-lstm"

Start training in the background: batch size 16, and each model's own learning rate from the paper (3e-5 for the LSTM baseline, 1e-4 for the NTMs). About 30 minutes for 1M sequences.

Fewer, larger batches means fewer weight updates, so if the cost falls too slowly, raise the rate: add `--learning-rate 3e-4`. Add `--sequences` to stop earlier.

In [ ]:
!nohup python -u main.py train --model {MODEL} --batch-size 16 > train_{MODEL}.log 2>&1 &
!sleep 30 && tail -n 3 train_{MODEL}.log

Check progress. Re-run whenever you like: it shows the latest cost and elapsed minutes.

In [ ]:
!tail -n 5 train_{MODEL}.log

Draw Figures 3 and 5 from the latest saved model. Works any time after the first 1,000 sequences (the first log line).

In [ ]:
!python main.py plot --model {MODEL}
import os
from IPython.display import Image, display
for path in [f'figures/{MODEL}_learning_curve.png', f'figures/{MODEL}_generalisation.png']:
    if os.path.exists(path):
        display(Image(path))

Try your own sequence. Edit `vectors` below: each item is 8 bits of 0/1. Or set `random_length` to a number to use a random sequence of that length (try something longer than 20 to see the LSTM fail). Works during training too, with the latest saved model.

In [ ]:
vectors = "10110010 01100101 11110000 00001111"
random_length = None  # e.g. 30

if random_length:
    !python main.py try --model {MODEL} --random {random_length}
else:
    !python main.py try --model {MODEL} {vectors}
from IPython.display import Image, display
display(Image(f'figures/{MODEL}_try.png'))

**Memory use (the paper's Figure 6).** NTM only. Shows, timestep by timestep: the input, the vectors written to memory and the write weightings on the left; the output, the vectors read back and the read weightings on the right. A network that has learned to copy shows a diagonal stripe in both weightings — the write head stepping along memory while reading the input, then the read head retracing the same locations.

In [ ]:
!python main.py memory --model {MODEL} --length 20
from IPython.display import Image, display
display(Image(f'figures/{MODEL}_memory_length20.png'))

Optional: copy the results and figures to Google Drive, so they survive when the Colab session ends.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/ntm-results
!cp -r results figures /content/drive/MyDrive/ntm-results/

## Leave it running overnight

Mount Drive first (the cell below this one), then run this. It trains both models — the NTM on the CPU and the LSTM on the GPU with the paper's settings — and redraws every figure from the latest checkpoints every 15 minutes, copying figures and results to Drive each time. Whatever has finished by morning is already plotted, and nothing is lost if Colab disconnects.

In [ ]:
!nohup bash overnight.sh > overnight.log 2>&1 &
!sleep 45 && tail -n 5 overnight.log train_ntm-ff.log

In the morning: progress, then the figures.

In [ ]:
!tail -n 3 overnight.log train_ntm-ff.log train_lstm.log
from IPython.display import Image, display
import os
for f in ['ntm-ff_learning_curve', 'ntm-ff_generalisation', 'ntm-ff_memory_length20', 'ntm-ff_memory_length40',
          'lstm_learning_curve', 'lstm_generalisation']:
    if os.path.exists(f'figures/{f}.png'):
        print(f)
        display(Image(f'figures/{f}.png'))